In [1]:
from pyspark.sql import SparkSession  
import getpass 
username = getpass 
spark = SparkSession.builder.config('spark.ui.port','0').\
        config('spark.sql.warehouse.dir',f"/user/itv022063/warehouse").\
        enableHiveSupport().\
        master('yarn').\
        getOrCreate()

In [2]:
spark.conf.set("spark.sql.unacceptable_rated_pts", 0)
spark.conf.set("spark.sql.very_bad_rated_pts", 100)
spark.conf.set("spark.sql.bad_rated_pts", 250)
spark.conf.set("spark.sql.good_rated_pts", 500)
spark.conf.set("spark.sql.very_good_rated_pts", 650)
spark.conf.set("spark.sql.excellent_rated_pts", 800)

In [3]:
spark.conf.set("spark.sql.unacceptable_grade_pts", 750)
spark.conf.set("spark.sql.very_bad_grade_pts", 1000)
spark.conf.set("spark.sql.bad_grade_pts", 1500)
spark.conf.set("spark.sql.good_grade_pts", 2000)
spark.conf.set("spark.sql.very_good_grade_pts", 2500)

In [4]:
bad_customer_data_final_df = spark.read \
.format("csv") \
.option("header", True) \
.option("inferSchema", True) \
.load("/user/itv022063/Yash/bad_data/bad_customer_data_final")

In [5]:
bad_customer_data_final_df.createOrReplaceTempView("bad_data_customer")

In [6]:
ph_df=spark.sql("""
SELECT
    l.member_id,
    CASE
        WHEN p.last_payment_amount < (l.monthly_installment * 0.5)
            THEN ${spark.sql.very_bad_rated_pts}

        WHEN p.last_payment_amount > (l.monthly_installment * 0.5)
             AND p.last_payment_amount < l.monthly_installment
            THEN ${spark.sql.bad_rated_pts}

        WHEN p.last_payment_amount = l.monthly_installment
            THEN ${spark.sql.good_rated_pts}

        WHEN p.last_payment_amount > l.monthly_installment
             AND p.last_payment_amount <= (l.monthly_installment * 1.5)
            THEN ${spark.sql.very_good_rated_pts}

        WHEN p.last_payment_amount > (l.monthly_installment * 1.5)
            THEN ${spark.sql.excellent_rated_pts}

        ELSE ${spark.sql.unacceptable_rated_pts}
    END AS last_payment_pts,

    CASE
        WHEN p.total_payment_received >= (l.funded_amount * 0.5)
            THEN ${spark.sql.very_good_grade_pts}

        WHEN p.total_payment_received < (l.funded_amount * 0.5)
             AND p.total_payment_received > 0
            THEN ${spark.sql.good_grade_pts}

        WHEN p.total_payment_received = 0
             OR p.total_payment_received IS NULL
            THEN ${spark.sql.unacceptable_grade_pts}
    END AS total_payment_pts

FROM itv022063_lending_club.loans_repayments p
INNER JOIN itv022063_lending_club.loans l
    ON l.loan_id = p.loan_id

WHERE l.member_id NOT IN (
    SELECT member_id FROM bad_data_customer
)
""")

In [7]:
ph_df.createOrReplaceTempView("ph_pts")

In [19]:
spark.sql("select * from ph_pts limit 5")

member_id,last_payment_pts,total_payment_pts
7c0d246380da9bde4...,500,2500
d1e691a8aa0a65598...,500,2500
938bcca7023b54b72...,500,2500
c1dfe602b07dbba63...,500,2500
6c3e36bdb6659d60f...,800,2500


In [9]:
ldh_ph_df=spark.sql("""
SELECT 
     p.*,

     CASE
         WHEN d.delinq_2yrs = 0
              THEN ${spark.sql.excellent_rated_pts}
         WHEN d.delinq_2yrs BETWEEN 1 AND 2
              THEN ${spark.sql.bad_rated_pts}
         WHEN d.delinq_2yrs BETWEEN 3 AND 5
              THEN ${spark.sql.very_bad_rated_pts}
         WHEN d.delinq_2yrs > 5 OR d.delinq_2yrs IS NULL
              THEN ${spark.sql.bad_rated_pts}
     END AS delinq_2yrs_pts,

     CASE
         WHEN l.pub_rec = 0
              THEN ${spark.sql.excellent_rated_pts}
         WHEN l.pub_rec BETWEEN 1 AND 2
              THEN ${spark.sql.bad_rated_pts}
         WHEN l.pub_rec BETWEEN 3 AND 5
              THEN ${spark.sql.very_bad_rated_pts}
         WHEN l.pub_rec > 5 OR l.pub_rec IS NULL
              THEN ${spark.sql.bad_rated_pts}
     END AS public_records_pts,

     CASE
         WHEN l.pub_rec_bankruptcies = 0
              THEN ${spark.sql.excellent_rated_pts}
         WHEN l.pub_rec_bankruptcies BETWEEN 1 AND 2
              THEN ${spark.sql.bad_rated_pts}
         WHEN l.pub_rec_bankruptcies BETWEEN 3 AND 5
              THEN ${spark.sql.very_bad_rated_pts}
         WHEN l.pub_rec_bankruptcies > 5 OR l.pub_rec_bankruptcies IS NULL
              THEN ${spark.sql.bad_rated_pts}
     END AS public_bankruptcies_pts,

     CASE
         WHEN l.inq_last_6mths = 0
              THEN ${spark.sql.excellent_rated_pts}
         WHEN l.inq_last_6mths BETWEEN 1 AND 2
              THEN ${spark.sql.bad_rated_pts}
         WHEN l.inq_last_6mths BETWEEN 3 AND 5
              THEN ${spark.sql.very_bad_rated_pts}
         WHEN l.inq_last_6mths > 5 OR l.inq_last_6mths IS NULL
              THEN ${spark.sql.bad_rated_pts}
     END AS enq_pts

FROM itv022063_lending_club.loans_defaulters_detail_rec_enq_new l
INNER JOIN 
      itv022063_lending_club.loans_defaulters_delinq_new d
       ON 
          d.member_id = l.member_id
INNER JOIN ph_pts p
       ON p.member_id = l.member_id
WHERE l.member_id NOT IN (
     SELECT member_id FROM bad_data_customer
)
""")

In [10]:
ldh_ph_df.createOrReplaceTempView("ldh_ph_df")

In [11]:
fh_ldh_ph_df = spark.sql("""
SELECT
    ldef.*,

    /* ---------- Loan Status Points ---------- */
    CASE
        WHEN LOWER(l.loan_status) = 'fully paid'
            THEN ${spark.sql.excellent_rated_pts}
        WHEN LOWER(l.loan_status) = 'current'
            THEN ${spark.sql.good_rated_pts}
        WHEN LOWER(l.loan_status) = 'in grace period'
            THEN ${spark.sql.bad_rated_pts}
        WHEN LOWER(l.loan_status) IN ('late (16-30 days)', 'late (31-120 days)')
            THEN ${spark.sql.very_bad_rated_pts}
        WHEN LOWER(l.loan_status) = 'charged off'
            THEN ${spark.sql.unacceptable_rated_pts}
        ELSE ${spark.sql.unacceptable_rated_pts}
    END AS loan_status_pts,

    /* ---------- Home Ownership Points ---------- */
    CASE
        WHEN LOWER(a.home_ownership) = 'own'
            THEN ${spark.sql.excellent_rated_pts}
        WHEN LOWER(a.home_ownership) = 'rent'
            THEN ${spark.sql.good_rated_pts}
        WHEN LOWER(a.home_ownership) = 'mortgage'
            THEN ${spark.sql.bad_rated_pts}
        WHEN LOWER(a.home_ownership) = 'any'
             OR a.home_ownership IS NULL
            THEN ${spark.sql.very_bad_rated_pts}
    END AS home_pts,

    /* ---------- Credit Utilization Points ---------- */
    CASE
        WHEN l.funded_amount <= a.total_high_credit_limit * 0.10
            THEN ${spark.sql.excellent_rated_pts}
        WHEN l.funded_amount <= a.total_high_credit_limit * 0.20
            THEN ${spark.sql.very_good_rated_pts}
        WHEN l.funded_amount <= a.total_high_credit_limit * 0.30
            THEN ${spark.sql.good_rated_pts}
        WHEN l.funded_amount <= a.total_high_credit_limit * 0.50
            THEN ${spark.sql.bad_rated_pts}
        WHEN l.funded_amount <= a.total_high_credit_limit * 0.70
            THEN ${spark.sql.very_bad_rated_pts}
        ELSE ${spark.sql.unacceptable_rated_pts}
    END AS credit_limit_pts,

    /* ---------- Grade & Subgrade Points ---------- */
    CASE
        /* Grade A */
        WHEN a.sub_grade = 'A1' THEN ${spark.sql.excellent_rated_pts}
        WHEN a.sub_grade = 'A2' THEN ${spark.sql.excellent_rated_pts} * 0.95
        WHEN a.sub_grade = 'A3' THEN ${spark.sql.excellent_rated_pts} * 0.90
        WHEN a.sub_grade = 'A4' THEN ${spark.sql.excellent_rated_pts} * 0.85
        WHEN a.sub_grade = 'A5' THEN ${spark.sql.excellent_rated_pts} * 0.80

        /* Grade B */
        WHEN a.sub_grade = 'B1' THEN ${spark.sql.very_good_rated_pts}
        WHEN a.sub_grade = 'B2' THEN ${spark.sql.very_good_rated_pts} * 0.95
        WHEN a.sub_grade = 'B3' THEN ${spark.sql.very_good_rated_pts} * 0.90
        WHEN a.sub_grade = 'B4' THEN ${spark.sql.very_good_rated_pts} * 0.85
        WHEN a.sub_grade = 'B5' THEN ${spark.sql.very_good_rated_pts} * 0.80

        /* Grade C */
        WHEN a.sub_grade = 'C1' THEN ${spark.sql.good_rated_pts}
        WHEN a.sub_grade = 'C2' THEN ${spark.sql.good_rated_pts} * 0.95
        WHEN a.sub_grade = 'C3' THEN ${spark.sql.good_rated_pts} * 0.90
        WHEN a.sub_grade = 'C4' THEN ${spark.sql.good_rated_pts} * 0.85
        WHEN a.sub_grade = 'C5' THEN ${spark.sql.good_rated_pts} * 0.80

        /* Grade D */
        WHEN a.sub_grade = 'D1' THEN ${spark.sql.bad_rated_pts}
        WHEN a.sub_grade = 'D2' THEN ${spark.sql.bad_rated_pts} * 0.95
        WHEN a.sub_grade = 'D3' THEN ${spark.sql.bad_rated_pts} * 0.90
        WHEN a.sub_grade = 'D4' THEN ${spark.sql.bad_rated_pts} * 0.85
        WHEN a.sub_grade = 'D5' THEN ${spark.sql.bad_rated_pts} * 0.80

        /* Grade E */
        WHEN a.sub_grade = 'E1' THEN ${spark.sql.very_bad_rated_pts}
        WHEN a.sub_grade = 'E2' THEN ${spark.sql.very_bad_rated_pts} * 0.95
        WHEN a.sub_grade = 'E3' THEN ${spark.sql.very_bad_rated_pts} * 0.90
        WHEN a.sub_grade = 'E4' THEN ${spark.sql.very_bad_rated_pts} * 0.85
        WHEN a.sub_grade = 'E5' THEN ${spark.sql.very_bad_rated_pts} * 0.80

        /* Grade F & G */
        WHEN a.grade IN ('F', 'G')
            THEN ${spark.sql.unacceptable_rated_pts}
    END AS grade_pts

FROM ldh_ph_df ldef
INNER JOIN itv022063_lending_club.loans l
    ON ldef.member_id = l.member_id
INNER JOIN itv022063_lending_club.customers_new a
    ON a.member_id = ldef.member_id

WHERE ldef.member_id NOT IN (
    SELECT member_id FROM bad_data_customer
)
""")

In [12]:
fh_ldh_ph_df.createOrReplaceTempView("fh_ldh_ph_pts")

In [20]:
spark.sql("select * from fh_ldh_ph_pts limit 5")

member_id,last_payment_pts,total_payment_pts,delinq_2yrs_pts,public_records_pts,public_bankruptcies_pts,enq_pts,loan_status_pts,home_pts,credit_limit_pts,grade_pts
0000036e9afe019a6...,800,2500,250,800,800,100,800,250,800,500.00
00138bdf80cc69bc6...,800,2500,250,800,800,800,800,500,650,500.00
0025c8d102d846a73...,250,2500,250,800,800,800,800,250,800,450.00
002a3d2b85c722148...,800,2500,250,800,800,800,800,500,800,400.00
006073596a788a4b0...,500,2000,100,800,800,800,500,250,650,400.00


In [14]:
loan_score = spark.sql("""
select member_id, ((last_payment_pts+total_payment_pts)*0.20) as payment_history_pts,
                 ((delinq_2yrs_pts + public_records_pts + public_bankruptcies_pts + enq_pts) * 0.45) as defaulters_history_pts,
                 ((loan_status_pts + home_pts + credit_limit_pts + grade_pts)*0.35) as financial_health_pts
                 from fh_ldh_ph_pts
                 
""")

In [22]:
final_loan_score = loan_score.withColumn('loan_score', loan_score.payment_history_pts + loan_score.defaulters_history_pts + loan_score.financial_health_pts)

In [23]:
final_loan_score.show()

+--------------------+-------------------+----------------------+--------------------+----------+
|           member_id|payment_history_pts|defaulters_history_pts|financial_health_pts|loan_score|
+--------------------+-------------------+----------------------+--------------------+----------+
|000c8875b71a6b47c...|             660.00|               1192.50|            885.5000| 2738.0000|
|003769d7f54c7859e...|             500.00|               1192.50|            402.5000| 2095.0000|
|003e1e6cbd2920bbb...|             600.00|                697.50|            766.5000| 2064.0000|
|004017b21bd4d6271...|             520.00|               1192.50|            822.5000| 2535.0000|
|005b4c3db3fce07dc...|             600.00|                697.50|            619.5000| 1917.0000|
|00710707c563c2119...|             660.00|               1192.50|            829.5000| 2682.0000|
|007da79904f69970d...|             660.00|               1192.50|            822.5000| 2675.0000|
|00f435a80d0440ece..

In [24]:
final_loan_score.createOrReplaceTempView("loan_score_eval")

In [28]:
loan_score_final = spark.sql("""
    select ls.* ,
             CASE
             WHEN loan_score> ${spark.sql.very_good_grade_pts} THEN 'A'
             WHEN loan_score<= ${spark.sql.very_good_grade_pts} AND loan_score> ${spark.sql.good_grade_pts} THEN 'B'
             WHEN loan_score<= ${spark.sql.good_grade_pts} AND loan_score> ${spark.sql.bad_grade_pts} THEN 'C'
             WHEN loan_score<= ${spark.sql.bad_grade_pts} AND loan_score> ${spark.sql.very_bad_grade_pts} THEN 'D'
             WHEN loan_score<= ${spark.sql.very_bad_grade_pts} AND loan_score> ${spark.sql.unacceptable_grade_pts} THEN 'E'
             WHEN loan_score<= ${spark.sql.unacceptable_grade_pts} THEN 'F'
             END as loan_final_grade
    from loan_score_eval ls
""")

In [31]:
loan_score_final.createOrReplaceTempView("loan_score_final_table")

In [32]:
spark.sql("select * from loan_score_final_table where loan_final_grade in ('C')")

member_id,payment_history_pts,defaulters_history_pts,financial_health_pts,loan_score,loan_final_grade
005b4c3db3fce07dc...,600.00,697.50,619.5000,1917.0000,C
017d1fd3d6169ee29...,560.00,697.50,694.7500,1952.2500,C
01d0c48835e969a01...,500.00,1192.50,262.5000,1955.0000,C
021a6ce1b67f3bc81...,500.00,450.00,758.6250,1708.6250,C
04a56cd37ddf106b4...,500.00,945.00,507.5000,1952.5000,C
0514f7ad4030ba481...,500.00,877.50,341.2500,1718.7500,C
06e451d3fcd652b6b...,600.00,697.50,691.2500,1988.7500,C
0813f246eeb0d971a...,630.00,810.00,245.0000,1685.0000,C
09a632bd9a4d62f1b...,600.00,945.00,402.5000,1947.5000,C
09f32bedf17f27478...,600.00,877.50,411.2500,1888.7500,C


In [33]:
loan_score_final.write \
.format("parquet") \
.mode("overwrite") \
.option("path", "/user/itv022063/Yash/processed/loans_score") \
.save()

In [34]:
spark.stop()